# NCHU Rental — 階段④ 泛化重訓閉環 (Colab)

**完整閉環:** 957 多樣 query append → 重訓 bi-encoder → T3 query encoder ONNX → T4 重算 property embeddings → baseline vs 重訓後 **Recall@30 Δ**。

**前置:** 先 merge PR #55(同源修正:property 文字用 `text` field)再跑,否則訓練資料分布偏移、Δ 不可信。

**Runtime:** 選 GPU(T4 即可;A100 更快,bf16/tf32 自動偵測)。

**設計約束(對齊 spec `docs/spec/generalization-data.md`):**
- 第一輪 **append 求穩**:957 pair 併進 `recommendation_train.json`,不取代原 29443 筆。
- **同源**:訓練 property 文字 = `text` field = 上線 embedding field(PR #55 已對齊)。
- **同源 caveat**:評估 query 與訓練 query 皆生成,Recall Δ 僅作相對趨勢判讀,不宣稱絕對泛化。

## Cell 0 — 確認 GPU

In [ ]:
import torch
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('bf16 支援:', torch.cuda.is_bf16_supported())
else:
    print('⚠️ 沒有 GPU — Runtime → Change runtime type → GPU')

## Cell 1 — Clone / Pull Repo
改成你的 repo URL;已 clone 則自動 pull 最新 main(含 PR #54 + #55)。

In [ ]:
import os

REPO_URL = 'https://github.com/eric20041027/nchu-edge-rental-ai.git'
REPO_DIR = '/content/nchu-edge-rental-ai'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git checkout main && git pull

os.chdir(REPO_DIR)
print('工作目錄:', os.getcwd())
# 確認同源修正已在(PR #55):field 註解
!grep -m1 'field=.text' frontend/assets/property_embeddings.json > /dev/null 2>&1 || python3 -c "import json; print('embedding field =', json.load(open('frontend/assets/property_embeddings.json'))['field'])"

## Cell 2 — 安裝依賴

In [ ]:
!pip install -q \
    transformers \
    tokenizers \
    onnx \
    onnxruntime \
    numpy
print('依賴安裝完成')

## Cell 3 — Append 957 多樣 query 進訓練集(第一輪求穩)

把 `generalization_queries.json`(957 pair)併進 `recommendation_train.json`(原 29443)。
**append 不取代**:寫到新檔 `recommendation_train_stage4.json`,原檔不動,可隨時回退。
schema 一致(`query`/`property`/`label`/`is_hard`/`relevance`),`_load_pairs` 直接吃。

In [ ]:
import json

BASE = 'data/processed/recommendation_train.json'
GEN  = 'data/processed/generalization_queries.json'
OUT  = 'data/processed/recommendation_train_stage4.json'

base = json.load(open(BASE, encoding='utf-8'))
gen  = json.load(open(GEN,  encoding='utf-8'))

# 對齊既有 schema:只留 _load_pairs 用到的欄位(去掉生成器側錄的 src_idx/feat)。
KEEP = {'query', 'property', 'label', 'relevance', 'is_hard'}
gen_clean = [{k: r[k] for k in KEEP if k in r} for r in gen]

merged = base + gen_clean
json.dump(merged, open(OUT, 'w', encoding='utf-8'), ensure_ascii=False)

pos_gen  = sum(1 for r in gen_clean if str(r.get('label')) == '1')
hard_gen = sum(1 for r in gen_clean if str(r.get('is_hard')).lower() == 'true')
print(f'原訓練 {len(base)} + 階段④ {len(gen_clean)}(正 {pos_gen}/硬負 {hard_gen}) = {len(merged)} 筆')
print(f'寫入 {OUT}(原 {BASE} 不動,可回退)')

# 同源自檢:階段④ property 文字應全落在 property_data.text
pd = json.load(open('frontend/assets/property_data.json', encoding='utf-8'))
text_set = {p.get('text') for p in pd}
hit = sum(1 for r in gen_clean if r['property'] in text_set)
print(f'同源自檢:{hit}/{len(gen_clean)} 階段④ property 落在 property_data.text', '✅' if hit == len(gen_clean) else '⚠️ 未對齊!先 merge PR #55')

## Cell 4 — 超參數(bi-encoder 專屬,對齊 colab_train_bi_encoder.ipynb)

In [ ]:
import os

# 關鍵:TRAIN_DATA_PATH 指向 append 後的合併檔(config 用此 env 覆寫預設)。
os.environ['TRAIN_DATA_PATH']        = 'data/processed/recommendation_train_stage4.json'
os.environ['BI_ENCODER_SAVED_DIR']   = '/content/saved_models/rbt6_bi_encoder_stage4'
os.environ['MODEL_CHECKPOINT']       = 'hfl/rbt6'   # CE 同源 base
os.environ['RANDOM_SEED']            = '42'
os.environ['BI_ENCODER_TEMPERATURE'] = '0.05'       # MNRL scale≈20

EPOCHS     = 5
BATCH_SIZE = 64
LR         = 2e-5
MAX_LENGTH = 64
USE_BF16   = True   # T4 不支援自動回退 fp32
USE_TF32   = True
print('超參數設定完成 | 訓練資料:', os.environ['TRAIN_DATA_PATH'])

## Cell 5 — 重訓 bi-encoder(InfoNCE / MNRL)

In [ ]:
bf16_flag = '--bf16' if USE_BF16 else ''
tf32_flag = '--tf32' if USE_TF32 else ''
cmd = (
    f"python3 -m pipeline.model_training.train_bi_encoder "
    f"--epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LR} "
    f"--max-length {MAX_LENGTH} {bf16_flag} {tf32_flag} "
    f"--output-dir {os.environ['BI_ENCODER_SAVED_DIR']}"
)
print(cmd)
!{cmd}

## Cell 6 — Baseline Recall@30(重訓**前** = 現有上線向量)

重訓前先用**現有** `property_embeddings.json` 算評估集 Recall@30 → baseline。
harness Recall 半吃 `tests/fixtures/generalization_eval.json`(13 筆,GT 資料算)。

In [ ]:
# 重訓前:現有上線向量 + 現有 query encoder ONNX → baseline
print('===== BASELINE(重訓前)=====')
!python3 tests/eval_generalization.py --k 30 || echo '若缺 onnxruntime,上一格 Cell 2 已裝;若缺現有 query ONNX,確認 frontend/models 有 bi_encoder query 模型'

## Cell 7 — T3:重訓後 query encoder → ONNX(+ INT8 量化)

export 重訓的 query 編碼路徑(mean-pool + L2-norm 在圖內)。

In [ ]:
!python3 -m pipeline.model_training.export_bi_encoder \
    --saved-dir {os.environ['BI_ENCODER_SAVED_DIR']} \
    --max-length {MAX_LENGTH}
print('T3 query encoder ONNX export 完成')

## Cell 8 — T4:用重訓模型重算 property embeddings

**同源鐵則:`--field text`** 對齊上線 + 訓練資料 field。dtype float16(半檔案大小)。
覆寫 `frontend/assets/property_embeddings.json`。

In [ ]:
!python3 -m pipeline.data_prep.build_property_embeddings \
    --saved-dir {os.environ['BI_ENCODER_SAVED_DIR']} \
    --field text \
    --dtype float16 \
    --batch-size 64
print('T4 property embeddings 重算完成')
import json
e = json.load(open('frontend/assets/property_embeddings.json'))
print(f"新 embedding: model={e['model']} dim={e['dim']} count={e['count']} field={e['field']}")

## Cell 9 — 重訓後 Recall@30 + Δ

用**重訓的 query ONNX + 重算的 property 向量**再跑一次評估集 → 比 baseline。
Recall↑ = 多樣 query 重訓改善了泛化召回(相對 Δ,同源 caveat 在)。

In [ ]:
print('===== 重訓後 =====')
!python3 tests/eval_generalization.py --k 30
print()
print('↑ 與 Cell 6 baseline 比較:Recall@30 上升即第一輪泛化重訓有效。')
print('  下一步(本機 preview 親驗):把 holdout 7 筆餵進前端 recommend() 看實際召回對不對。')

## Cell 10 — 備份產物到 Google Drive(強烈建議)

備份重訓權重 + 重算 embedding + query ONNX,避免 Colab session 過期丟失。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DEST = '/content/drive/MyDrive/nchu_stage4_backup'
os.makedirs(DEST, exist_ok=True)

# 重算的前端產物 + 重訓權重
shutil.copy('frontend/assets/property_embeddings.json', f'{DEST}/property_embeddings.json')
if os.path.exists(os.environ['BI_ENCODER_SAVED_DIR']):
    shutil.make_archive(f'{DEST}/rbt6_bi_encoder_stage4', 'zip', os.environ['BI_ENCODER_SAVED_DIR'])
# query ONNX(T3 產;路徑依 export_bi_encoder 預設,通常在 frontend/models 或 saved_dir)
for cand in ['frontend/models', os.environ['BI_ENCODER_SAVED_DIR']]:
    for root, _, files in os.walk(cand):
        for fn in files:
            if fn.endswith('.onnx') and 'bi' in fn.lower():
                shutil.copy(os.path.join(root, fn), f'{DEST}/{fn}')
print('備份完成 →', DEST)
print('下載 frontend/assets/property_embeddings.json + query ONNX 換進前端即上線。')

## Cell 11 — 快速下載重訓產物(直接觸發瀏覽器下載)

把 T4 重算的 `property_embeddings.json` + T3 重訓的 query ONNX 打包成一個 zip,
直接觸發瀏覽器下載 — 不必進 Google Drive 找檔。下載後解壓,把兩個檔覆蓋到
**專案根目錄**(`Desktop/nchu-edge-rental-ai/`),再請 Claude 換進前端跑 preview 驗。

In [ ]:
import os, zipfile
from google.colab import files

# 兩個確定路徑(T4 / T3 產物)
EMB  = 'frontend/assets/property_embeddings.json'
ONNX = 'frontend/models/bi_encoder_dir/bi_encoder_quant.onnx'

missing = [p for p in (EMB, ONNX) if not os.path.exists(p)]
if missing:
    raise SystemExit(f'缺檔(先跑 Cell 7 T3 + Cell 8 T4):{missing}')

ZIP = 'stage4_retrained.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(EMB,  arcname='property_embeddings.json')
    z.write(ONNX, arcname='bi_encoder_quant.onnx')

size_mb = os.path.getsize(ZIP) / 1e6
print(f'打包完成 {ZIP}  ({size_mb:.1f} MB)= property_embeddings.json + bi_encoder_quant.onnx')
print('觸發下載中...解壓後把兩個檔覆蓋到專案根目錄 Desktop/nchu-edge-rental-ai/')
files.download(ZIP)